In [1]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'birkbeck.dat', 'cleaned_IMDb_reviews.csv', 'Customer Review Cleaning System.ipynb', 'IMDB Dataset.csv', 'Search_Query_Spelling_Corrector.ipynb.ipynb']


In [2]:
import re
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
file_path = "birkbeck.dat"

with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Dataset loaded successfully!")
print("Total lines:", len(lines))

Dataset loaded successfully!
Total lines: 42269


In [4]:
for line in lines[:20]:
    print(line.strip())

$Albert
Ab
$America
Ameraca
Amercia
$American
Ameracan
$April
Apirl
$Austrian
Austrain
$Badcock's
badcock
$Bechuanaland
bechuarnia_land
$Botswana
botuania
$Cambridge
Cambrige
$Canada


In [5]:
vocabulary = set()
error_pairs = []

current_correct_word = None

for line in lines:

    line = line.strip()

    # Ignore empty lines
    if not line:
        continue

    # Correct word starts with $
    if line.startswith("$"):

        current_correct_word = line[1:].lower()
        vocabulary.add(current_correct_word)

    # Misspelled word
    else:

        misspelled_word = line.lower()

        if current_correct_word:
            error_pairs.append(
                (misspelled_word, current_correct_word)
            )

print("Number of correct words:", len(vocabulary))
print("Number of spelling-error pairs:", len(error_pairs))

Number of correct words: 6130
Number of spelling-error pairs: 36133


In [6]:
vocabulary_df = pd.DataFrame(
    sorted(vocabulary),
    columns=["correct_word"]
)

vocabulary_df.head(20)

,correct_word
0,a
1,a-quiver
2,a_bit
3,a_few
4,a_little
5,a_long
6,a_look
7,a_lot
8,a_museum
9,abattoir


In [7]:
error_df = pd.DataFrame(
    error_pairs,
    columns=["incorrect_word", "correct_word"]
)

error_df.head(20)

,incorrect_word,correct_word
0,ab,albert
1,ameraca,america
2,amercia,america
3,ameracan,american
4,apirl,april
5,austrain,austrian
6,badcock,badcock's
7,bechuarnia_land,bechuanaland
8,botuania,botswana
9,cambrige,cambridge


In [8]:
print("Number of correct words:", len(vocabulary))
print("Number of spelling-error pairs:", len(error_pairs))

Number of correct words: 6130
Number of spelling-error pairs: 36133


In [9]:
query = input("Enter your search query: ")

print("\nOriginal Query:")
print(query)

Enter your search query:  machne lerning cours



Original Query:
machne lerning cours


In [10]:
def tokenize_query(query):
    query = query.lower()
    tokens = re.findall(r"[a-zA-Z]+", query)
    return tokens

In [11]:
tokens = tokenize_query(query)

print("Tokenized Query:")
print(tokens)

Tokenized Query:
['machne', 'lerning', 'cours']


In [12]:
incorrect_words = []

for word in tokens:
    if word not in vocabulary:
        incorrect_words.append(word)

print("Incorrect Words:")
print(incorrect_words)

Incorrect Words:
['machne', 'lerning', 'cours']


In [13]:
def edit_distance(word1, word2):

    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = np.zeros((rows, cols), dtype=int)

    # Initialize first column
    for i in range(rows):
        matrix[i][0] = i

    # Initialize first row
    for j in range(cols):
        matrix[0][j] = j

    # Calculate edit distance
    for i in range(1, rows):
        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,       # deletion
                matrix[i][j - 1] + 1,       # insertion
                matrix[i - 1][j - 1] + cost # substitution
            )

    return matrix[rows - 1][cols - 1]

In [14]:
print("machne → machine:",
      edit_distance("machne", "machine"))

print("lerning → learning:",
      edit_distance("lerning", "learning"))

machne → machine: 1
lerning → learning: 1


In [15]:
def find_closest_word(word):

    best_word = None
    best_distance = float("inf")

    for candidate in vocabulary:

        # Ignore candidates that are very different in length
        if abs(len(word) - len(candidate)) > 2:
            continue

        distance = edit_distance(word, candidate)

        if distance < best_distance:
            best_distance = distance
            best_word = candidate

    return best_word, best_distance

In [16]:
word = "machne"

corrected_word, distance = find_closest_word(word)

print("Incorrect Word:", word)
print("Suggested Correction:", corrected_word)
print("Edit Distance:", distance)

Incorrect Word: machne
Suggested Correction: machine
Edit Distance: 1


In [17]:
def correct_query(query):

    # Tokenize query
    tokens = tokenize_query(query)

    corrected_tokens = []
    incorrect_words = []
    suggestions = []

    for word in tokens:

        # If word is already correct
        if word in vocabulary:

            corrected_tokens.append(word)

        else:

            incorrect_words.append(word)

            corrected_word, distance = find_closest_word(word)

            corrected_tokens.append(corrected_word)

            suggestions.append(
                (word, corrected_word, distance)
            )

    corrected_query = " ".join(corrected_tokens)

    return tokens, incorrect_words, suggestions, corrected_query

In [18]:
tokens, incorrect_words, suggestions, corrected_query = correct_query(query)

print("==========================================")
print("SEARCH QUERY SPELLING CORRECTOR")
print("==========================================")

print("\nOriginal Query:")
print(query)

print("\nTokenized Query:")
print(tokens)

print("\nIncorrect Words:")
print(incorrect_words)

print("\nSuggested Corrections:")

for wrong, correct, distance in suggestions:
    print(f"{wrong} → {correct} (Edit Distance: {distance})")

print("\nFinal Corrected Query:")
print(corrected_query)

SEARCH QUERY SPELLING CORRECTOR

Original Query:
machne lerning cours

Tokenized Query:
['machne', 'lerning', 'cours']

Incorrect Words:
['machne', 'lerning', 'cours']

Suggested Corrections:
machne → machine (Edit Distance: 1)
lerning → learning (Edit Distance: 1)
cours → court (Edit Distance: 1)

Final Corrected Query:
machine learning court


In [19]:
test_queries = [
    "machne lerning cours",
    "computr scince",
    "programing languge",
    "artifical inteligence",
    "databse managment"
]

for query in test_queries:

    tokens, incorrect_words, suggestions, corrected_query = correct_query(query)

    print("\n==========================================")

    print("Original Query:")
    print(query)

    print("\nIncorrect Words:")
    print(incorrect_words)

    print("\nSuggested Corrections:")

    for wrong, correct, distance in suggestions:
        print(f"{wrong} → {correct} (Distance: {distance})")

    print("\nFinal Corrected Query:")
    print(corrected_query)


Original Query:
machne lerning cours

Incorrect Words:
['machne', 'lerning', 'cours']

Suggested Corrections:
machne → machine (Distance: 1)
lerning → learning (Distance: 1)
cours → court (Distance: 1)

Final Corrected Query:
machine learning court

Original Query:
computr scince

Incorrect Words:
['computr', 'scince']

Suggested Corrections:
computr → complete (Distance: 3)
scince → science (Distance: 1)

Final Corrected Query:
complete science

Original Query:
programing languge

Incorrect Words:
['programing', 'languge']

Suggested Corrections:
programing → programmes (Distance: 3)
languge → language (Distance: 1)

Final Corrected Query:
programmes language

Original Query:
artifical inteligence

Incorrect Words:
['artifical', 'inteligence']

Suggested Corrections:
artifical → artificial (Distance: 1)
inteligence → intelligence (Distance: 1)

Final Corrected Query:
artificial intelligence

Original Query:
databse managment

Incorrect Words:
['databse', 'managment']

Suggested Corre

In [20]:
results = []

for query in test_queries:

    tokens, incorrect_words, suggestions, corrected_query = correct_query(query)

    results.append({
        "Original Query": query,
        "Incorrect Words": ", ".join(incorrect_words),
        "Corrected Query": corrected_query
    })

results_df = pd.DataFrame(results)

results_df

,Original Query,Incorrect Words,Corrected Query
0,machne lerning cours,"machne, lerning, cours",machine learning court
1,computr scince,"computr, scince",complete science
2,programing languge,"programing, languge",programmes language
3,artifical inteligence,"artifical, inteligence",artificial intelligence
4,databse managment,"databse, managment",table management
